# CaT-GNN: Causal Temporal Graph Neural Network for Fraud Detection

## 📌 Executive Summary & Architecture Overview
Financial fraud detection on transaction networks faces a critical challenge: **spurious correlations and distribution shifts**. Graph Neural Networks (GNNs) often overfit to transient transaction patterns (e.g., promotional spikes, temporary merchant hotspots) that do not represent true causal fraud indicators.

**CaT-GNN** solves this through **causal invariance** and **temporal graph attention**:
1. **Harmonic Time Encoding ($Time2Vec$)**: Periodic, continuous time representation mapping time gaps $\Delta t$ across multiple frequency scales.
2. **Multi-Head Relational Attention**: Captures temporal dynamics across multiple edge relations (`Source`, `Target`, `Location`, `Type`).
3. **Parameter-Free Causal Inspector**: Uses attention weights to separate local neighborhoods into **Causal Nodes** $\mathcal{C}_i$ vs **Environment Nodes** $\mathcal{E}_i$.
4. **Counterfactual Beta-Mixup Intervention**: Replaces environment value vectors with a Beta-mixup of top-$k$ causal donor representations ($do(V_{\text{env}})$).
5. **Causal-Invariant Objective**: Jointly minimizes task loss on original and counterfactual subgraphs.

## 1. Setup & Dependencies

In [ ]:
import math
import os
from collections import defaultdict
from collections.abc import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch
import torch.nn.functional as F
import torchmetrics
from sklearn.metrics import (
    auc,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    roc_curve,
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, Dataset

pl.seed_everything(42, workers=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__} | Device: {device}")

## 2. Temporal Multi-Window Feature Engineering & Graph Dataset

Transactions are processed into:
- **122-dimensional continuous feature maps**: Multi-window statistical aggregations (mean, sum, std, bias, count, distinct entities) across 15 expanding time spans $[2, 3, 5, \dots, 24000]$ seconds.
- **Categorical entity embeddings**: Mappings for `Location` and `Type`.
- **Multi-relational temporal adjacency**: Dynamic neighborhood structures ordered backward in time.

In [ ]:
def build_features(csv_path: str, cache_path: str = "sffsd_ai4risk_preprocessed.pt") -> dict:
    """Extract 122-dim temporal multi-window features and relational graph adjacency."""
    if os.path.exists(cache_path):
        print(f"Loading cached features from {cache_path}...")
        return torch.load(cache_path, weights_only=False)

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Dataset not found at {csv_path}.")

    df = pd.read_csv(csv_path).sort_values("Time").reset_index(drop=True)
    num_nodes = len(df)
    time_spans = [2, 3, 5, 15, 20, 50, 100, 150, 200, 300, 864, 2590, 5100, 10000, 24000]
    num_spans = len(time_spans)

    fe_matrix = np.zeros((num_nodes, num_spans * 8 + 2), dtype=np.float32)
    times = df["Time"].values
    amts = df["Amount"].values.astype(np.float32)

    fe_matrix[:, 0] = amts
    fe_matrix[:, 1] = np.log1p(np.maximum(0.0, amts))

    tgt_codes = pd.factorize(df["Target"])[0]
    loc_codes = pd.factorize(df["Location"])[0]
    type_codes = pd.factorize(df["Type"])[0]

    source_to_indices = defaultdict(list)
    for idx, s in enumerate(df["Source"].values):
        source_to_indices[s].append(idx)

    # Multi-window statistical features: [avg, tot, std, bias, count, num_tgt, num_loc, num_type]
    for s, idxs in source_to_indices.items():
        if len(idxs) == 1:
            i = idxs[0]
            amt_i = amts[i]
            for s_idx in range(num_spans):
                base_col = 2 + s_idx * 8
                fe_matrix[i, base_col + 0] = amt_i
                fe_matrix[i, base_col + 1] = amt_i
                fe_matrix[i, base_col + 2] = 0.0
                fe_matrix[i, base_col + 3] = 0.0
                fe_matrix[i, base_col + 4] = 1.0
                fe_matrix[i, base_col + 5] = 1.0
                fe_matrix[i, base_col + 6] = 1.0
                fe_matrix[i, base_col + 7] = 1.0
        else:
            idxs = np.array(idxs, dtype=np.int32)
            group_times = times[idxs]
            group_amts = amts[idxs]
            group_tgts = tgt_codes[idxs]
            group_locs = loc_codes[idxs]
            group_types = type_codes[idxs]
            m = len(idxs)

            for s_idx, length in enumerate(time_spans):
                base_col = 2 + s_idx * 8
                left_bounds = np.searchsorted(group_times, group_times - length, side="left")
                for pos in range(m):
                    orig_idx = idxs[pos]
                    lb = left_bounds[pos]
                    window_amts = group_amts[lb : pos + 1]
                    cnt = len(window_amts)
                    tot = float(np.sum(window_amts))
                    avg = tot / cnt
                    std = float(np.std(window_amts)) if cnt > 1 else 0.0
                    bias = float(group_amts[pos] - avg)

                    num_tgt = float(len(np.unique(group_tgts[lb : pos + 1])))
                    num_loc = float(len(np.unique(group_locs[lb : pos + 1])))
                    num_type = float(len(np.unique(group_types[lb : pos + 1])))

                    fe_matrix[orig_idx, base_col + 0] = avg
                    fe_matrix[orig_idx, base_col + 1] = tot
                    fe_matrix[orig_idx, base_col + 2] = std
                    fe_matrix[orig_idx, base_col + 3] = bias
                    fe_matrix[orig_idx, base_col + 4] = float(cnt)
                    fe_matrix[orig_idx, base_col + 5] = num_tgt
                    fe_matrix[orig_idx, base_col + 6] = num_loc
                    fe_matrix[orig_idx, base_col + 7] = num_type

    scaler = StandardScaler()
    scaled_cont = scaler.fit_transform(fe_matrix)

    cat_cols = ["Location", "Type"]
    cat_dims = []
    for col in cat_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        cat_dims.append(len(le.classes_))

    # Construct temporal multi-relational edges
    adj_list = [[] for _ in range(num_nodes)]
    pair = ["Source", "Target", "Location", "Type"]
    for rel_id, column in enumerate(pair):
        entity_to_txs = defaultdict(list)
        for tx_idx, entity_id in enumerate(df[column].values):
            entity_to_txs[entity_id].append(tx_idx)
        for tx_indices in entity_to_txs.values():
            if len(tx_indices) > 1:
                for idx_pos, i in enumerate(tx_indices):
                    prior_txs = tx_indices[max(0, idx_pos - 10) : idx_pos]
                    for p in prior_txs:
                        adj_list[i].append((p, rel_id))

    times_raw = df["Time"].values
    for i in range(num_nodes):
        seen = set()
        deduped = []
        sorted_neighbors = sorted(adj_list[i], key=lambda item: times_raw[item[0]], reverse=True)
        for nbr, rel in sorted_neighbors:
            if nbr not in seen:
                seen.add(nbr)
                deduped.append((nbr, rel))
        adj_list[i] = deduped

    labeled_indices = np.where(df["Labels"].isin([0, 1]).values)[0]

    data_dict = {
        "X_cont": torch.tensor(scaled_cont, dtype=torch.float32),
        "X_cat": torch.tensor(df[cat_cols].values, dtype=torch.long),
        "Y": torch.tensor(df["Labels"].values, dtype=torch.float32),
        "adj": adj_list,
        "times": torch.tensor(times_raw, dtype=torch.float32),
        "cat_dims": cat_dims,
        "cont_dim": scaled_cont.shape[1],
        "labeled_indices": labeled_indices,
    }
    torch.save(data_dict, cache_path)
    return data_dict


class SFFSDGraphDataset(Dataset):
    def __init__(self, data_dict: dict, split_indices: np.ndarray, max_neighbors: int = 20):
        self.max_neighbors = max_neighbors
        self.indices = split_indices
        self.X_cont = data_dict["X_cont"]
        self.X_cat = data_dict["X_cat"]
        self.Y = data_dict["Y"]
        self.adj_list = data_dict["adj"]
        self.times = data_dict["times"]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int):
        node_id = self.indices[idx]
        x_target_cont = self.X_cont[node_id]
        x_target_cat = self.X_cat[node_id]
        y = self.Y[node_id]
        t_target = self.times[node_id]

        neighbors = self.adj_list[node_id]
        sampled = neighbors[: self.max_neighbors]
        actual_M = len(sampled)

        x_neighbor_cont = torch.zeros(self.max_neighbors, self.X_cont.shape[1])
        x_neighbor_cat = torch.zeros(self.max_neighbors, self.X_cat.shape[1], dtype=torch.long)
        t_gap = torch.zeros(self.max_neighbors)
        rel_ids = torch.full((self.max_neighbors,), 4, dtype=torch.long)
        valid_mask = torch.zeros(self.max_neighbors, dtype=torch.bool)

        if actual_M > 0:
            sampled_nodes = [item[0] for item in sampled]
            sampled_rels = [item[1] for item in sampled]
            sampled_idx = np.array(sampled_nodes, dtype=np.int64)

            x_neighbor_cont[:actual_M] = self.X_cont[sampled_idx]
            x_neighbor_cat[:actual_M] = self.X_cat[sampled_idx]
            t_gap[:actual_M] = torch.abs(t_target - self.times[sampled_idx])
            rel_ids[:actual_M] = torch.tensor(sampled_rels, dtype=torch.long)
            valid_mask[:actual_M] = True

        return x_target_cont, x_target_cat, x_neighbor_cont, x_neighbor_cat, t_gap, rel_ids, valid_mask, y


class FraudGraphDataModule(pl.LightningDataModule):
    def __init__(self, data_path: str = "S-FFSD.csv", max_neighbors: int = 20, train_ratio: float = 0.7, batch_size: int = 128, num_workers: int = 2):
        super().__init__()
        self.save_hyperparameters()
        self.data_dict = None

    def setup(self, stage: str | None = None):
        if self.data_dict is None:
            self.data_dict = build_features(self.hparams.data_path)
            self.cont_dim = self.data_dict["cont_dim"]
            self.cat_dims = self.data_dict["cat_dims"]
            labeled_idx = self.data_dict["labeled_indices"]
            train_end = int(len(labeled_idx) * self.hparams.train_ratio)

            self.train_idx = labeled_idx[:train_end]
            self.test_idx = labeled_idx[train_end:]
            train_labels = self.data_dict["Y"][self.train_idx]
            num_pos = train_labels.sum().item()
            num_neg = len(train_labels) - num_pos
            self.pos_weight = num_neg / max(num_pos, 1.0)

        self.train_ds = SFFSDGraphDataset(self.data_dict, self.train_idx, self.hparams.max_neighbors)
        self.test_ds = SFFSDGraphDataset(self.data_dict, self.test_idx, self.hparams.max_neighbors)

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.hparams.batch_size, shuffle=True, num_workers=self.hparams.num_workers)

    def val_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.hparams.batch_size, shuffle=False, num_workers=self.hparams.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.hparams.batch_size, shuffle=False, num_workers=self.hparams.num_workers)

## 3. Harmonic Time Encoder & Multi-Head Relational Attention

### Time2Vec Formulation:
$$\Phi(\Delta t) = \cos(\mathbf{w} \cdot \Delta t + \mathbf{b})$$

### Temporal Attention Scoring:
$$e_{ij}^h = \text{LeakyReLU}\left(\mathbf{a}_h^T [\mathbf{q}_i^h \,;\, \mathbf{k}_j^h \,;\, \Phi(\Delta t_{ij})] \cdot \frac{1}{\sqrt{2 d_h + d_{\text{time}}}}\right) + \text{rel\_bias}(r)^h$$

In [ ]:
class HarmonicTimeEncoder(nn.Module):
    """Time2Vec harmonic encoding: Phi(dt) = cos(w * dt + b)."""
    def __init__(self, time_dim: int):
        super().__init__()
        self.time_dim = time_dim
        init_w = 1.0 / (10.0 ** torch.linspace(0.0, 9.0, time_dim))
        self.w = nn.Parameter(init_w.float())
        self.b = nn.Parameter(torch.zeros(time_dim))

    def forward(self, delta_t: torch.Tensor) -> torch.Tensor:
        phase = delta_t.unsqueeze(-1) * self.w + self.b
        return torch.cos(phase)


class TemporalAttentionLayer(nn.Module):
    """Multi-head relational temporal attention layer."""
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        heads: int = 4,
        time_dim: int = 16,
        num_relations: int = 4,
        dropout: float = 0.1,
        negative_slope: float = 0.2,
    ):
        super().__init__()
        assert hidden_dim % heads == 0
        self.heads = heads
        self.d_h = hidden_dim // heads
        self.hidden_dim = hidden_dim
        self.time_dim = time_dim

        self.time_encoder = HarmonicTimeEncoder(time_dim)
        self.W_q = nn.Linear(in_dim, hidden_dim, bias=False)
        self.W_k = nn.Linear(in_dim, hidden_dim, bias=False)
        self.W_v = nn.Linear(in_dim, hidden_dim, bias=False)
        self.rel_embed = nn.Embedding(num_relations + 1, heads)
        self.attn = nn.Parameter(torch.empty(heads, 2 * self.d_h + time_dim))
        nn.init.xavier_uniform_(self.attn)

        self.scale = 1.0 / math.sqrt(2 * self.d_h + time_dim)
        self.leaky_relu = nn.LeakyReLU(negative_slope)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_target, x_neighbor, t_gap, valid_mask, rel_ids=None):
        N, M, _in_dim = x_neighbor.shape
        H, d_h = self.heads, self.d_h

        t_emb = self.time_encoder(t_gap)  # [N, M, time_dim]
        Q = self.W_q(x_target).view(N, 1, H, d_h)
        K = self.W_k(x_neighbor).view(N, M, H, d_h)
        V = self.W_v(x_neighbor).view(N, M, H, d_h)

        Q_exp = Q.expand(-1, M, -1, -1)
        t_emb_exp = t_emb.unsqueeze(2).expand(-1, -1, H, -1)
        concat = torch.cat([Q_exp, K, t_emb_exp], dim=-1)

        e = self.leaky_relu((concat * self.attn).sum(dim=-1) * self.scale)
        if rel_ids is not None:
            e = e + self.rel_embed(rel_ids)

        e = e.masked_fill(~valid_mask.unsqueeze(-1), float("-inf"))
        alpha = torch.softmax(e, dim=1)
        alpha = torch.nan_to_num(alpha, nan=0.0)
        alpha = self.dropout(alpha)
        return alpha, V

    def aggregate(self, alpha: torch.Tensor, V: torch.Tensor) -> torch.Tensor:
        N, _M, H = alpha.shape
        z = (alpha.unsqueeze(-1) * V).sum(dim=1)
        return z.reshape(N, H * self.d_h)

## 4. Causal Neighborhood Inspection & Beta-Mixup Intervention

- **Causal Inspector**: Identifies environment slots $\mathcal{E}_i$ by selecting the lowest-importance attention weights up to ratio $r_e$.
- **Causal Intervener**: Replaces $V_j$ ($j \in \mathcal{E}_i$) with a Beta-mixup with high-importance causal donor vectors $x_c \in \mathcal{C}_i$.

In [ ]:
class CausalInspector(nn.Module):
    """Split neighborhood into causal vs environment nodes via attention weights."""
    def __init__(self, env_ratio: float = 0.2):
        super().__init__()
        assert 0.0 < env_ratio < 1.0
        self.env_ratio = env_ratio

    def forward(self, alpha: torch.Tensor, valid_mask: torch.Tensor):
        _N, _M, _H = alpha.shape
        importance = alpha.mean(dim=2)  # Mean across attention heads [N, M]
        valid_counts = valid_mask.sum(dim=1)

        masked_scores = importance.masked_fill(~valid_mask, float("inf"))
        order = torch.argsort(masked_scores, dim=1)
        rank = torch.argsort(order, dim=1)

        raw_env = torch.ceil(self.env_ratio * valid_counts.float()).long()
        max_allowed_env = torch.clamp(valid_counts - 1, min=0)
        env_count = torch.min(raw_env, max_allowed_env)

        env_mask = (rank < env_count.unsqueeze(1)) & valid_mask
        causal_mask = valid_mask & (~env_mask)
        return env_mask, causal_mask, importance


class CausalIntervener(nn.Module):
    """Counterfactual intervention: do(V_env) = lambda * V_env + (1 - lambda) * V_causal."""
    def __init__(self, top_k: int = 5, beta_alpha: float = 2.0, beta_beta: float = 2.0):
        super().__init__()
        self.top_k = top_k
        self.beta_alpha = beta_alpha
        self.beta_beta = beta_beta

    def forward(self, V: torch.Tensor, importance: torch.Tensor, env_mask: torch.Tensor, causal_mask: torch.Tensor):
        N, M, H, d_h = V.shape
        k = min(self.top_k, M)

        causal_scores = importance.masked_fill(~causal_mask, float("-inf"))
        topk_scores, topk_idx = torch.topk(causal_scores, k=k, dim=1)
        valid_topk = topk_scores > float("-inf")

        no_causal_row = ~causal_mask.any(dim=1)
        col_probs = valid_topk.float()
        col_probs = torch.where(no_causal_row.unsqueeze(1), torch.full_like(col_probs, 1.0 / k), col_probs)
        col_probs = col_probs / col_probs.sum(dim=1, keepdim=True).clamp(min=1e-12)

        samples = torch.multinomial(col_probs, num_samples=M, replacement=True)
        selected_idx = torch.gather(topk_idx, 1, samples)
        x_c = torch.gather(V, 1, selected_idx.view(N, M, 1, 1).expand(-1, -1, H, d_h))

        beta_dist = torch.distributions.Beta(self.beta_alpha, self.beta_beta)
        lam = beta_dist.sample((N, M)).to(device=V.device, dtype=V.dtype)
        lam_b = lam.view(N, M, 1, 1)

        mixed = lam_b * V + (1.0 - lam_b) * x_c
        V_intervened = torch.where(env_mask.view(N, M, 1, 1), mixed, V)
        V_intervened = torch.where(no_causal_row.view(N, 1, 1, 1), V, V_intervened)
        return V_intervened, lam

## 5. Causal Invariant Loss Objectives

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{task}}(\hat{y}, y) + \gamma \cdot \mathcal{L}_{\text{task}}(\hat{y}_{\text{int}}, y) + \eta \|\mathbf{w}\|_2^2$$

In [ ]:
class FocalLoss(nn.Module):
    """Binary Focal Loss: FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)."""
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        targets = targets.float()
        p = torch.sigmoid(logits)
        ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p_t = p * targets + (1.0 - p) * (1.0 - targets)
        alpha_t = self.alpha * targets + (1.0 - self.alpha) * (1.0 - targets)
        loss = alpha_t * (1.0 - p_t).pow(self.gamma) * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()


class CompositeLoss(nn.Module):
    def __init__(self, base_loss: str = "focal", gamma: float = 1.0, eta: float = 0.0, focal_alpha: float = 0.25, focal_gamma: float = 2.0, pos_weight: float | None = None):
        super().__init__()
        self.task_loss = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        self.gamma = gamma
        self.eta = eta

    def forward(self, logits: torch.Tensor, targets: torch.Tensor, logits_intervened: torch.Tensor | None = None, model_parameters: Iterable[torch.nn.Parameter] | None = None):
        l_task = self.task_loss(logits, targets)
        total = l_task
        components = {"task_loss": l_task.detach()}

        if logits_intervened is not None:
            l_inv = self.task_loss(logits_intervened, targets)
            total = total + self.gamma * l_inv
            components["invariant_loss"] = l_inv.detach()

        if self.eta > 0.0 and model_parameters is not None:
            l2 = sum(p.pow(2).sum() for p in model_parameters if p.requires_grad)
            total = total + self.eta * l2
            components["l2_penalty"] = l2.detach()

        components["total_loss"] = total.detach()
        return total, components

## 6. Complete CaT-GNN Model Architecture

In [ ]:
class CaTGNN(nn.Module):
    def __init__(
        self,
        cont_dim: int,
        cat_dims: list[int],
        emb_dim: int = 16,
        hidden_dim: int = 64,
        heads: int = 4,
        time_dim: int = 16,
        num_relations: int = 4,
        env_ratio: float = 0.2,
        top_k: int = 5,
        beta_alpha: float = 2.0,
        beta_beta: float = 2.0,
        mlp_hidden: int = 64,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_classes, emb_dim) for num_classes in cat_dims
        ])
        in_dim = cont_dim + (len(cat_dims) * emb_dim)
        self.target_proj = nn.Linear(in_dim, hidden_dim)

        self.attn_layer = TemporalAttentionLayer(
            in_dim=in_dim,
            hidden_dim=hidden_dim,
            heads=heads,
            time_dim=time_dim,
            num_relations=num_relations,
            dropout=dropout,
        )
        self.inspector = CausalInspector(env_ratio=env_ratio)
        self.intervener = CausalIntervener(top_k=top_k, beta_alpha=beta_alpha, beta_beta=beta_beta)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, mlp_hidden),
            nn.LayerNorm(mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def encode_inputs(self, x_cont: torch.Tensor, x_cat: torch.Tensor) -> torch.Tensor:
        if len(self.embeddings) > 0 and x_cat.numel() > 0:
            embs = [emb(x_cat[..., i]) for i, emb in enumerate(self.embeddings)]
            return torch.cat([x_cont] + embs, dim=-1)
        return x_cont

    def forward(
        self,
        x_target_cont: torch.Tensor,
        x_target_cat: torch.Tensor,
        x_neighbor_cont: torch.Tensor,
        x_neighbor_cat: torch.Tensor,
        t_gap: torch.Tensor,
        valid_mask: torch.Tensor,
        rel_ids: torch.Tensor | None = None,
        intervene: bool = True,
    ) -> dict[str, torch.Tensor]:
        x_target = self.encode_inputs(x_target_cont, x_target_cat)
        x_neighbor = self.encode_inputs(x_neighbor_cont, x_neighbor_cat)

        h_target = torch.relu(self.target_proj(x_target))
        alpha, V = self.attn_layer(x_target, x_neighbor, t_gap, valid_mask, rel_ids=rel_ids)
        z_neigh = self.attn_layer.aggregate(alpha, V)

        h_fused = torch.cat([h_target, z_neigh], dim=-1)
        logits = self.classifier(h_fused).squeeze(-1)
        out = {"logits": logits, "z": h_fused, "alpha": alpha}

        if intervene:
            env_mask, causal_mask, importance = self.inspector(alpha, valid_mask)
            V_int, lam = self.intervener(V, importance, env_mask, causal_mask)
            z_int = self.attn_layer.aggregate(alpha, V_int)
            h_fused_int = torch.cat([h_target, z_int], dim=-1)
            logits_int = self.classifier(h_fused_int).squeeze(-1)
            out.update({
                "logits_intervened": logits_int,
                "z_intervened": h_fused_int,
                "env_mask": env_mask,
                "causal_mask": causal_mask,
                "lambda": lam,
            })
        return out

## 7. PyTorch Lightning Module & Training Workflow

In [ ]:
class CaTGNNLightningModule(pl.LightningModule):
    def __init__(
        self,
        cont_dim: int,
        cat_dims: list[int],
        emb_dim: int = 16,
        hidden_dim: int = 64,
        heads: int = 4,
        time_dim: int = 16,
        num_relations: int = 4,
        env_ratio: float = 0.2,
        top_k: int = 5,
        beta_alpha: float = 2.0,
        beta_beta: float = 2.0,
        mlp_hidden: int = 64,
        dropout: float = 0.1,
        base_loss: str = "focal",
        gamma: float = 1.0,
        warmup_epochs: int = 3,
        lr: float = 1e-3,
        weight_decay: float = 1e-5,
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = CaTGNN(
            cont_dim=cont_dim,
            cat_dims=cat_dims,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            heads=heads,
            time_dim=time_dim,
            num_relations=num_relations,
            env_ratio=env_ratio,
            top_k=top_k,
            beta_alpha=beta_alpha,
            beta_beta=beta_beta,
            mlp_hidden=mlp_hidden,
            dropout=dropout,
        )
        self.loss_fn = CompositeLoss(base_loss=base_loss, gamma=gamma)
        self.val_auroc = torchmetrics.AUROC(task="binary")
        self.val_auprc = torchmetrics.AveragePrecision(task="binary")
        self.test_auroc = torchmetrics.AUROC(task="binary")
        self.test_auprc = torchmetrics.AveragePrecision(task="binary")
        self.val_probs, self.val_targets = [], []
        self.test_probs, self.test_targets = [], []

    def current_gamma(self) -> float:
        warmup = self.hparams.warmup_epochs
        if warmup <= 0 or self.current_epoch >= warmup:
            return self.hparams.gamma
        return self.hparams.gamma * (self.current_epoch / max(1, warmup))

    def shared_step(self, batch, intervene: bool):
        x_t_cont, x_t_cat, x_n_cont, x_n_cat, t_gap, rel_ids, valid_mask, y = batch
        out = self.model(
            x_target_cont=x_t_cont,
            x_target_cat=x_t_cat,
            x_neighbor_cont=x_n_cont,
            x_neighbor_cat=x_n_cat,
            t_gap=t_gap,
            valid_mask=valid_mask,
            rel_ids=rel_ids,
            intervene=intervene,
        )
        self.loss_fn.gamma = self.current_gamma() if intervene else self.hparams.gamma
        loss, components = self.loss_fn(out["logits"], y, logits_intervened=out.get("logits_intervened"))
        probs = torch.sigmoid(out["logits"])
        return loss, components, probs, y

    def training_step(self, batch, batch_idx):
        loss, components, _, y = self.shared_step(batch, intervene=True)
        self.log_dict({f"train/{k}": v for k, v in components.items()}, on_epoch=True, prog_bar=True, batch_size=y.size(0))
        return loss

    def validation_step(self, batch, batch_idx):
        loss, _components, probs, y = self.shared_step(batch, intervene=False)
        y_int = y.long()
        self.val_auroc.update(probs, y_int)
        self.val_auprc.update(probs, y_int)
        self.val_probs.append(probs.detach().cpu())
        self.val_targets.append(y_int.detach().cpu())
        return loss

    def on_validation_epoch_end(self):
        self.log("val/auroc", self.val_auroc.compute(), prog_bar=True)
        self.log("val/auprc", self.val_auprc.compute(), prog_bar=True)
        self.val_auroc.reset()
        self.val_auprc.reset()
        self.val_probs.clear()
        self.val_targets.clear()

    def test_step(self, batch, batch_idx):
        loss, _components, probs, y = self.shared_step(batch, intervene=False)
        y_int = y.long()
        self.test_auroc.update(probs, y_int)
        self.test_auprc.update(probs, y_int)
        self.test_probs.append(probs.detach().cpu())
        self.test_targets.append(y_int.detach().cpu())
        return loss

    def on_test_epoch_end(self):
        self.log("test/auroc", self.test_auroc.compute())
        self.log("test/auprc", self.test_auprc.compute())
        if self.test_probs:
            probs = torch.cat(self.test_probs).numpy()
            targets = torch.cat(self.test_targets).numpy()
            best_f1, best_thresh = 0.0, 0.5
            for t in np.linspace(0.02, 0.98, 97):
                score = float(f1_score(targets, (probs >= t).astype(int), average="macro", zero_division=0))
                if score > best_f1:
                    best_f1, best_thresh = score, float(t)
            self.log("test/f1_macro", best_f1)
            self.log("test/threshold", best_thresh)
        self.test_auroc.reset()
        self.test_auprc.reset()
        self.test_probs.clear()
        self.test_targets.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs, eta_min=1e-5)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "epoch"}}

## 8. Model Training & Evaluation

In [ ]:
# Prepare DataModule
dm = FraudGraphDataModule(data_path="S-FFSD.csv", max_neighbors=20, train_ratio=0.7, batch_size=128, num_workers=2)
dm.setup()

module = CaTGNNLightningModule(
    cont_dim=dm.cont_dim,
    cat_dims=dm.cat_dims,
    emb_dim=16,
    hidden_dim=64,
    heads=4,
    time_dim=16,
    env_ratio=0.2,
    top_k=5,
    gamma=1.0,
    warmup_epochs=3,
    lr=1e-3,
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    enable_progress_bar=True,
    log_every_n_steps=10,
)

trainer.fit(module, datamodule=dm)
test_metrics = trainer.test(module, datamodule=dm)[0]

print("\n" + "=" * 50)
print("FINAL TEST PERFORMANCE")
print("=" * 50)
for k, v in test_metrics.items():
    print(f"{k:20s}: {v:.4f}")

## 9. Performance Visualization: ROC & Precision-Recall Curves

In [ ]:
module.eval()
all_probs, all_targets = [], []

with torch.no_grad():
    for batch in dm.test_dataloader():
        x_t_c, x_t_cat, x_n_c, x_n_cat, tg, rel, mask, y = batch
        out = module.model(
            x_target_cont=x_t_c,
            x_target_cat=x_t_cat,
            x_neighbor_cont=x_n_c,
            x_neighbor_cat=x_n_cat,
            t_gap=tg,
            valid_mask=mask,
            rel_ids=rel,
            intervene=False,
        )
        all_probs.append(torch.sigmoid(out["logits"]).cpu().numpy())
        all_targets.append(y.cpu().numpy())

y_prob = np.concatenate(all_probs)
y_true = np.concatenate(all_targets)

fpr, tpr, _ = roc_curve(y_true, y_prob)
precision, recall, _ = precision_recall_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)
pr_auc = average_precision_score(y_true, y_prob)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(fpr, tpr, color="#e74c3c", lw=2.5, label=f"CaT-GNN (AUROC = {roc_auc:.4f})")
ax1.plot([0, 1], [0, 1], color="gray", linestyle="--")
ax1.set_title("ROC Curve", fontsize=13, fontweight="bold")
ax1.set_xlabel("False Positive Rate", fontsize=11)
ax1.set_ylabel("True Positive Rate", fontsize=11)
ax1.legend(loc="lower right", fontsize=11)
ax1.grid(True, alpha=0.3)

ax2.plot(recall, precision, color="#2980b9", lw=2.5, label=f"CaT-GNN (AUPRC = {pr_auc:.4f})")
ax2.set_title("Precision-Recall Curve", fontsize=13, fontweight="bold")
ax2.set_xlabel("Recall", fontsize=11)
ax2.set_ylabel("Precision", fontsize=11)
ax2.legend(loc="lower left", fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()